In [2]:
from apps.transactionEngine import me
from tinydb import Query
from apps.database import transactionDb
from apps.helperFunctions import formatResponse

In [ ]:
def cancelTransaction(tId, uId):
    # First we will check for the presence of the transaction in the database

    transaction = Query()
    results = transactionDb.search((transaction.tId == tId))

    if len(results) == 0:
        # No such transaction exists
        return formatResponse(statusCode=404, description="No such transaction exists", resource="transaction", state="data:notfound")
    
    dbUid = results[0].get("uId")
    if uId != dbUid:
        # Not Authorized to perform this transaction
        return formatResponse(statusCode=401, description="Not Authorized to perform this transaction", resource="transaction", state="action:unauth")

    for result in results:
        status = transaction.get("status")
        if status == "COMPLETED" or status == "IN-COMPLETE":
            return formatResponse(statusCode=401, description="Transaction already complete.", resource="transaction", state="action:unauth")

    # All the updates will be done at the matching engine    
    stockId = results[0].get("stockId"); side = results[0].get("side")
    transactionRequest = {
        "action": "remove-transaction",
        "tId": tId,
        "side": side,
        "stockId": stockId
    }
    me.transactionQueue.put(transactionRequest)

    return formatResponse(statusCode=200, description="Cancellation Reuqest Submitted", resource="transaction", state="action:success")
      